[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Mineria-Datos/blob/main/notebooks/notebook_semana3_limpieza_datos.ipynb)
# Actividad práctica Semana 3
## Análisis exploratorio de datos (EDA)
## Limpieza básica de datos en Python

**Dataset:** `clientes_retail_video_limpieza.csv`  
**Objetivo:** demostrar un procedimiento breve para identificar y tratar problemas básicos de calidad de datos: valores nulos, duplicados, inconsistencias y valores fuera de rango.

> Este dataset es una versión reducida y con errores intencionales, creada solo para fines pedagógicos.


## 1. Importar librerías

Comenzamos cargando las librerías básicas que utilizaremos en el tutorial.


In [ ]:
import pandas as pd
import numpy as np

## 2. Cargar el dataset

En este paso debes subir el archivo CSV desde tu computador.  
Haz clic en “Seleccionar archivo” y elige el dataset proporcionado para la actividad.

In [ ]:
from google.colab import files

# Subir archivo desde el computador
uploaded = files.upload()

# Obtener el nombre del archivo subido
file_name = list(uploaded.keys())[0]

# Leer el CSV
df = pd.read_csv(file_name)

df

Saving clientes_retail_video_limpieza.csv to clientes_retail_video_limpieza.csv


,id_cliente,edad,frecuencia_compra,monto_total,canal_preferido,categoria_frecuente
0,1001,44.0,9,234668.0,Online,Tecnología
1,1002,36.0,13,293824.0,online,Hogar
2,1003,NaN,3,327931.0,Tienda,Vestuario
3,1004,56.0,5,394157.0,ONLINE,Tecnología
4,1005,35.0,10,310130.0,Tienda,Vestuario
5,1006,150.0,15,388110.0,Tienda,Vestuario
6,1007,NaN,7,329026.0,tienda,Tecnología
7,1008,47.0,13,NaN,Tienda,Deportes
8,1009,-5.0,7,188493.0,Online,Deportes
9,1010,45.0,7,299828.0,Tienda,Hogar


## 3. Revisar estructura general

Antes de limpiar, conviene mirar el tamaño del dataset, los tipos de datos y una primera descripción general.


In [ ]:
df.shape

(15, 6)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_cliente           15 non-null     int64  
 1   edad                 13 non-null     float64
 2   frecuencia_compra    15 non-null     int64  
 3   monto_total          14 non-null     float64
 4   canal_preferido      15 non-null     object 
 5   categoria_frecuente  15 non-null     object 
dtypes: float64(2), int64(2), object(2)
memory usage: 852.0+ bytes


In [ ]:
df.describe(include="all")

,id_cliente,edad,frecuencia_compra,monto_total,canal_preferido,categoria_frecuente
count,15.000000,13.000000,15.000000,14.000000,15,15
unique,NaN,NaN,NaN,NaN,6,4
top,NaN,NaN,NaN,NaN,Tienda,Vestuario
freq,NaN,NaN,NaN,NaN,9,7
mean,1007.333333,43.538462,8.066667,332561.571429,NaN,NaN
std,4.082483,35.340287,4.495500,105506.976236,NaN,NaN
min,1001.000000,-5.000000,-2.000000,150643.000000,NaN,NaN
25%,1004.500000,32.000000,6.000000,295325.000000,NaN,NaN
50%,1007.000000,36.000000,9.000000,319030.500000,NaN,NaN
75%,1010.500000,45.000000,10.500000,392645.250000,NaN,NaN


## 4. Identificar valores faltantes

Revisamos si existen datos ausentes en las columnas del dataset.


In [ ]:
df.isnull().sum()

,0
id_cliente,0
edad,2
frecuencia_compra,0
monto_total,1
canal_preferido,0
categoria_frecuente,0


### Tratamiento simple de valores faltantes

Para este ejemplo:
- reemplazaremos la edad faltante por la mediana;
- reemplazaremos el monto faltante por la mediana.

En un análisis real, esta decisión debe justificarse según el problema y el significado de la variable.


In [ ]:
df["edad"] = df["edad"].fillna(df["edad"].median())
df["monto_total"] = df["monto_total"].fillna(df["monto_total"].median())

df.isnull().sum()

,0
id_cliente,0
edad,0
frecuencia_compra,0
monto_total,0
canal_preferido,0
categoria_frecuente,0


## 5. Identificar registros duplicados

Los duplicados pueden alterar conteos, promedios y resultados del análisis.


In [ ]:
df.duplicated().sum()

np.int64(1)

In [ ]:
df[df.duplicated()]

,id_cliente,edad,frecuencia_compra,monto_total,canal_preferido,categoria_frecuente
14,1005,35.0,10,310130.0,Tienda,Vestuario


### Eliminar duplicados

En este caso eliminamos filas completamente duplicadas.


In [ ]:
df = df.drop_duplicates()
df.shape

(14, 6)

## 6. Revisar inconsistencias categóricas

Observamos los valores únicos de la variable `canal_preferido`. Esto permite detectar diferencias de escritura o codificación.


In [ ]:
df["canal_preferido"].unique()

array(['Online', 'online', 'Tienda', 'ONLINE', 'tienda', 'OnLine'],
      dtype=object)

### Estandarizar categorías

Unificamos distintas formas de escribir el canal de compra.


In [ ]:
df["canal_preferido"] = (
    df["canal_preferido"]
    .str.strip()
    .str.lower()
    .replace({
        "online": "Online",
        "onl": "Online",
        "web": "Online",
        "tienda": "Tienda",
        "presencial": "Tienda"
    })
)

df["canal_preferido"].unique()

/tmp/ipykernel_45058/137070880.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["canal_preferido"] = (


array(['Online', 'Tienda'], dtype=object)

In [ ]:
# A veces pandas lanza una advertencia cuando trabajamos sobre subconjuntos.
# No es un error, pero puede traer problemas después.
# Para evitar eso, hacemos una copia explícita del DataFrame.
df = df.copy()

In [ ]:
df["canal_preferido"] = (
    df["canal_preferido"]
    .str.strip()
    .str.lower()
    .replace({
        "online": "Online",
        "onl": "Online",
        "web": "Online",
        "tienda": "Tienda",
        "presencial": "Tienda"
    })
)

df["canal_preferido"].unique()

array(['Online', 'Tienda'], dtype=object)

## 7. Detectar valores fuera de rango

Revisamos valores que no tienen sentido para el contexto del problema.  
En este ejemplo:
- la edad debería estar entre 18 y 80 años;
- la frecuencia de compra no debería ser negativa.


In [ ]:
df[df["edad"].notna() & ((df["edad"] < 18) | (df["edad"] > 80))]

,id_cliente,edad,frecuencia_compra,monto_total,canal_preferido,categoria_frecuente
5,1006,150.0,15,388110.0,Tienda,Vestuario
8,1009,-5.0,7,188493.0,Online,Deportes


In [ ]:
df[df["frecuencia_compra"] < 0]

,id_cliente,edad,frecuencia_compra,monto_total,canal_preferido,categoria_frecuente
11,1012,32.0,-2,403990.0,Tienda,Vestuario


### Tratamiento de valores fuera de rango

Para este tutorial aplicaremos una decisión simple:
- reemplazar edades fuera de rango por la mediana de las edades válidas;
- reemplazar frecuencias negativas por 0.

En un proyecto real, estas decisiones deberían revisarse con más detalle y quedar documentadas.


In [ ]:
edad_valida = df.loc[(df["edad"] >= 18) & (df["edad"] <= 80), "edad"].median()

df.loc[(df["edad"] < 18) | (df["edad"] > 80), "edad"] = edad_valida
df.loc[df["frecuencia_compra"] < 0, "frecuencia_compra"] = 0

df

,id_cliente,edad,frecuencia_compra,monto_total,canal_preferido,categoria_frecuente
0,1001,44.0,9,234668.0,Online,Tecnología
1,1002,36.0,13,293824.0,Online,Hogar
2,1003,36.0,3,327931.0,Tienda,Vestuario
3,1004,56.0,5,394157.0,Online,Tecnología
4,1005,35.0,10,310130.0,Tienda,Vestuario
5,1006,36.0,15,388110.0,Tienda,Vestuario
6,1007,36.0,7,329026.0,Tienda,Tecnología
7,1008,47.0,13,319030.5,Tienda,Deportes
8,1009,36.0,7,188493.0,Online,Deportes
9,1010,45.0,7,299828.0,Tienda,Hogar


## 8. Verificar el resultado de la limpieza

Después de limpiar, volvemos a revisar el dataset. Esta etapa es importante porque permite confirmar que las transformaciones tuvieron el efecto esperado.


In [ ]:
print("Valores faltantes por columna:")
display(df.isnull().sum())

print("Duplicados:")
display(df.duplicated().sum())

print("Valores únicos en canal_preferido:")
display(df["canal_preferido"].unique())

print("Rango de edad:")
display(df["edad"].min(), df["edad"].max())

print("Frecuencia mínima:")
display(df["frecuencia_compra"].min())

Valores faltantes por columna:


,0
id_cliente,0
edad,0
frecuencia_compra,0
monto_total,0
canal_preferido,0
categoria_frecuente,0


Duplicados:


np.int64(0)

Valores únicos en canal_preferido:


...

Rango de edad:


18.0

56.0

Frecuencia mínima:


0

## 9. Guardar dataset limpio

Finalmente, guardamos una nueva versión del dataset. Esto permite conservar el archivo original y trabajar con una versión preparada para análisis posterior.


In [ ]:
df.to_csv("clientes_retail_video_limpieza_limpio.csv", index=False)
print("Archivo guardado: clientes_retail_video_limpieza_limpio.csv")

Archivo guardado: clientes_retail_video_limpieza_limpio.csv


## Cierre del tutorial

En este procedimiento se revisaron cuatro tareas básicas de limpieza de datos:

1. identificación y tratamiento de valores faltantes;  
2. detección y eliminación de duplicados;  
3. estandarización de categorías;  
4. revisión y corrección de valores fuera de rango.

Estos pasos no reemplazan el análisis completo, pero ayudan a preparar un dataset más confiable para las siguientes etapas del trabajo.
